# 미션 01. 생성 옵션 실험실

같은 로컬 LLM도 생성 옵션에 따라 답변의 창의성, 안정성, 반복 정도가 달라집니다.  
이번 미션에서는 **한 번에 하나의 조건만 바꾸는 실험**으로 각 옵션의 역할을 확인합니다.

## 완료 조건

- `temperature`, `top_p`, `top_k`, `repetition_penalty` 실험을 모두 실행한다.
- 결과를 근거로 작업별 추천 설정을 작성한다.
- seed별 결과를 비교해 **능력 보유**와 **안정적인 지시 수행**을 구분해서 설명한다.


## 0. 모델 준비

강의에서 사용한 `Qwen/Qwen3-0.6B`를 **CUDA GPU + FP16**으로 불러옵니다. CUDA가 보이지 않으면 CPU로 우회하지 않고 오류를 내도록 하여 환경 문제를 바로 확인합니다.

In [13]:
import time
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

model_id = "Qwen/Qwen3-0.6B"
if not torch.cuda.is_available():
    raise RuntimeError(
        "현재 Jupyter 커널에서 CUDA를 사용할 수 없습니다. "
        "강의 때 사용한 Python 커널/가상환경이 선택됐는지 확인하세요."
    )

device = "cuda"
dtype = torch.float16

tok = AutoTokenizer.from_pretrained(model_id)
model = AutoModelForCausalLM.from_pretrained(model_id, dtype=dtype).to(device)
model.eval()

print(f"준비 완료: {model_id} / {device} / {dtype}")
print("사용 GPU:", torch.cuda.get_device_name(torch.cuda.current_device()))

Loading weights: 100%|██████████| 311/311 [00:01<00:00, 274.99it/s]


준비 완료: Qwen/Qwen3-0.6B / cuda / torch.float16
사용 GPU: NVIDIA GeForce RTX 3050 Laptop GPU


## 1. 실험 도구 만들기

`seed`를 같게 두면 옵션의 영향을 비교하기 쉬워집니다. 속도는 참고용이며, 첫 실행은 준비 작업 때문에 더 느릴 수 있습니다.

In [14]:
def ask(prompt, seed=42, max_new_tokens=120, **gen_kwargs):
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

    inputs = tok.apply_chat_template(
        [{"role": "user", "content": prompt}],
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        enable_thinking=False,
    ).to(device)

    if device == "cuda":
        torch.cuda.synchronize()
    started = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            pad_token_id=tok.eos_token_id,
            **gen_kwargs,
        )
    if device == "cuda":
        torch.cuda.synchronize()
    elapsed = time.perf_counter() - started

    input_length = inputs["input_ids"].shape[1]
    new_tokens = output.shape[1] - input_length
    answer = tok.decode(output[0, input_length:], skip_special_tokens=True)
    return {
        "answer": answer.strip(),
        "new_tokens": new_tokens,
        "seconds": round(elapsed, 2),
        "tok_per_s": round(new_tokens / elapsed, 1),
    }


def run_cases(prompt, cases, seed=42):
    results = []
    for label, options in cases:
        result = ask(prompt, seed=seed, **options)
        results.append({"label": label, "options": options, **result})
        print(f"\n{'=' * 20} {label} {'=' * 20}")
        print("설정:", options)
        print(result["answer"])
        print(f"[{result['new_tokens']} tokens / {result['seconds']} sec / {result['tok_per_s']} tok/s]")
    return results

## 2. 기준점: Greedy decoding

`do_sample=False`이면 확률이 가장 높은 다음 토큰을 계속 선택합니다. 동일한 입력에는 대체로 동일한 결과가 나옵니다.

In [15]:
creative_prompt = (
    "점심을 주제로 '점'과 '심'으로 시작하는 이행시를 지어줘. "
    "각 행은 20자 이내로 쓰고, 설명 없이 이행시만 답해."
)

greedy_results = run_cases(
    creative_prompt,
    [("greedy", {"do_sample": False, "max_new_tokens": 80})],
)


==================== greedy ====================
설정: {'do_sample': False, 'max_new_tokens': 80}
점심을 주제로 '점'과 '심'으로 시작하는 이행시:  
점심을 주제로 '점'과 '심'으로 시작하는 이행시:  
점심을 주제로 '점'과 '심'으로 시작하는 이행시:
[63 tokens / 2.37 sec / 26.6 tok/s]


## 3. 실험 A: temperature

온도가 낮으면 유력한 토큰에 선택이 집중되고, 높으면 덜 유력한 토큰도 선택될 가능성이 커집니다. 다른 조건은 고정합니다.

> 관찰 포인트: 표현이 얼마나 새로워지는가? 형식 위반이나 어색한 표현도 늘어나는가?

In [16]:
temperature_cases = [
    ("temperature=0.2", {"do_sample": True, "temperature": 0.2, "top_p": 1.0, "top_k": 0}),
    ("temperature=0.7", {"do_sample": True, "temperature": 0.7, "top_p": 1.0, "top_k": 0}),
    ("temperature=1.2", {"do_sample": True, "temperature": 1.2, "top_p": 1.0, "top_k": 0}),
]
temperature_results = run_cases(creative_prompt, temperature_cases)


==================== temperature=0.2 ====================
설정: {'do_sample': True, 'temperature': 0.2, 'top_p': 1.0, 'top_k': 0}
점심을 주제로 '점'과 '심'으로 시작하는 이행시:  
점심을 주제로 '점'과 '심'으로 시작하는 이행시:  
점심을 주제로 '점'과 '심'으로 시작하는 이행시:
[63 tokens / 2.19 sec / 28.7 tok/s]

==================== temperature=0.7 ====================
설정: {'do_sample': True, 'temperature': 0.7, 'top_p': 1.0, 'top_k': 0}
1. 점심을 주제로 한 이행시  
2. 점심을 주제로 한 이행시  
3. 점심을 주제로 한 이행시
[42 tokens / 1.33 sec / 31.7 tok/s]

==================== temperature=1.2 ====================
설정: {'do_sample': True, 'temperature': 1.2, 'top_p': 1.0, 'top_k': 0}
1. 업·점심  
2. 찬·족일  
3. 버DER·식점  

이행시 참조하세요.
[32 tokens / 0.98 sec / 32.8 tok/s]


### 기록 A

**Q1. 가장 창의적이라고 느낀 설정**
> **`temperature=1.2`**  
> 다른 설정에서 나오지 않은 단어 조합이 등장했지만, 창의적이라기보다 무작위성이 커진 결과에 가까웠다.

**Q2. 형식을 가장 잘 지킨 설정**
> **정확히 지킨 설정은 없었다.**  
> 그나마 `temperature=0.7`의 문장이 읽기 쉬웠지만, 두 줄 대신 세 항목을 출력했고 같은 문장을 반복했다.

**Q3. 온도가 너무 높을 때 나타난 문제**
> `업·점심`, `버DER·식점`처럼 의미가 불분명하거나 다른 문자 체계가 섞인 표현이 나왔고, 이행시 규칙도 지키지 못했다.

**Q4. 내가 선택한 균형점과 이유**
> **`temperature=0.7`**  
> 낮은 온도의 완전한 문장 복사와 높은 온도의 무의미한 표현 사이에서 문장 자체의 자연스러움은 가장 나았다. 다만 이 모델과 프롬프트에서는 설정만으로 이행시 형식을 해결하지 못했다.

## 4. 실험 B: top_p

누적 확률이 `top_p`에 도달할 때까지의 후보만 남깁니다. 값이 작을수록 후보 범위가 좁아집니다. 이번에는 `temperature=0.8`로 고정합니다.

In [17]:
top_p_cases = [
    ("top_p=0.5", {"do_sample": True, "temperature": 0.8, "top_p": 0.5, "top_k": 0}),
    ("top_p=0.8", {"do_sample": True, "temperature": 0.8, "top_p": 0.8, "top_k": 0}),
    ("top_p=0.95", {"do_sample": True, "temperature": 0.8, "top_p": 0.95, "top_k": 0}),
]
top_p_results = run_cases(creative_prompt, top_p_cases)


==================== top_p=0.5 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 0.5, 'top_k': 0}
점심을 주제로 '점'과 '심'으로 시작하는 이행시:  
점심은 점심을 주제로 하는 이행시입니다.
[38 tokens / 1.15 sec / 33.1 tok/s]

==================== top_p=0.8 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 0.8, 'top_k': 0}
1. 점심을 주제로 한 이행시  
2. 점심을 주제로 한 이행시  
3. 점심을 주제로 한 이행시
[42 tokens / 1.29 sec / 32.5 tok/s]

==================== top_p=0.95 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 0.95, 'top_k': 0}
1. 점심을 주제로 한 이행시로, 점심을 주제로한 이행시입니다.  
2. 심심한 시간을 주제로한 이행시입니다.
[47 tokens / 1.51 sec / 31.1 tok/s]


### 기록 B

**Q1. `top_p`가 작을 때의 특징**
> **`top_p=0.5`에서는 안전하고 익숙한 표현만 선택해 요청 문장을 되풀이하거나 정의하는 답변이 나왔다.**  
> 후보 범위가 좁아 새로운 내용을 만들지 못했다.

**Q2. `top_p`가 커질 때 추가로 나타난 표현**
> `top_p=0.95`에서는 `심심한 시간을...`처럼 기존 결과에 없던 표현이 나왔고 두 번째 줄을 `심`으로 시작하려는 모습도 보였다. 하지만 첫 줄의 형식과 전체 내용은 여전히 부정확했다.

**Q3. temperature 실험과 비교했을 때 느낀 차이**
> temperature를 `1.2`까지 높였을 때는 표현 자체가 크게 흐트러졌지만, temperature를 `0.8`로 고정하고 top_p만 높였을 때는 후보의 다양성이 늘면서도 문장 형태는 비교적 유지됐다.

## 5. 실험 C: top_k

확률이 높은 상위 K개의 토큰만 후보로 남깁니다. `temperature=0.8`, `top_p=1.0`으로 고정합니다.

In [18]:
top_k_cases = [
    ("top_k=5", {"do_sample": True, "temperature": 0.8, "top_p": 1.0, "top_k": 5}),
    ("top_k=20", {"do_sample": True, "temperature": 0.8, "top_p": 1.0, "top_k": 20}),
    ("top_k=50", {"do_sample": True, "temperature": 0.8, "top_p": 1.0, "top_k": 50}),
]
top_k_results = run_cases(creative_prompt, top_k_cases)


==================== top_k=5 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 1.0, 'top_k': 5}
1. 점심을 주제로 한 이행시  
2. 점심을 주제로 한 이행시  
3. 점심을 주제로 한 이행시
[42 tokens / 1.34 sec / 31.4 tok/s]

==================== top_k=20 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 1.0, 'top_k': 20}
1. 점심을 주제로 한 이행시로, 점심을 주제로한 이행시입니다.  
2. 심심한 시간을 주제로한 이행시입니다.
[47 tokens / 1.48 sec / 31.7 tok/s]

==================== top_k=50 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 1.0, 'top_k': 50}
1. 점심을 주제로 한 이행시로, 점심을 주제로한 이행시입니다.  
2. 심심한 시간을 주제로한 이행시입니다.
[47 tokens / 1.5 sec / 31.4 tok/s]


### 기록 C

**Q1. 후보를 5개로 제한했을 때의 특징**
> **확률이 높은 소수의 표현에 갇혀 `점심을 주제로 한 이행시`라는 동일 문장을 세 번 반복했다.**

**Q2. `top_k=20`과 `top_k=50`의 차이가 분명했는가?**
> **이번 실행에서는 차이가 없었다.**  
> 같은 seed에서 두 설정의 출력이 완전히 같았으므로, 실제로 선택된 토큰들이 이미 상위 20개 후보 안에 있었던 것으로 해석할 수 있다.

**Q3. 이 프롬프트에 가장 적절하다고 판단한 값과 이유**
> 실험값 중에서는 **`top_k=20`**. `top_k=5`의 반복에서 벗어났고, `top_k=50`으로 넓혀도 추가 개선이 관찰되지 않았기 때문이다. 다만 어느 값도 이행시를 성공시키지는 못했다.

## 6. 실험 D: repetition_penalty

이미 나온 토큰이 다시 선택되는 것을 억제합니다. 너무 높이면 자연스러운 반복까지 막아 문장이 어색해질 수 있습니다.

In [19]:
repetition_prompt = (
    "로컬 LLM 학습 모임을 홍보하는 서로 다른 문구 6개를 만들어줘. "
    "각 문구는 한 줄로 작성해."
)
repetition_cases = [
    ("penalty=1.0", {"do_sample": True, "temperature": 0.8, "top_p": 0.9, "repetition_penalty": 1.0, "max_new_tokens": 180}),
    ("penalty=1.1", {"do_sample": True, "temperature": 0.8, "top_p": 0.9, "repetition_penalty": 1.1, "max_new_tokens": 180}),
    ("penalty=1.3", {"do_sample": True, "temperature": 0.8, "top_p": 0.9, "repetition_penalty": 1.3, "max_new_tokens": 180}),
]
repetition_results = run_cases(repetition_prompt, repetition_cases)


==================== penalty=1.0 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 0.9, 'repetition_penalty': 1.0, 'max_new_tokens': 180}
1. 로컬 LLM 학습 모임을 가족을 위한 대상으로 홍보합니다.  
2. 지역 내에서 LLM 학습을 위한 모임을 열어두고 있습니다.  
3. 로컬 LLM 학습을 위한 활동을 확장하고자 합니다.  
4. 지역 사회에서 LLM 학습을 위한 모임을 위한 전략을 수립하고 있습니다.  
5. 지역 내에서 LLM 학습을 위한 프로그램을 운영하고자 합니다.  
6. 로컬 LLM 학습을 위한 모임을 위해 활동을 시작하고 있습니다.
[138 tokens / 4.32 sec / 32.0 tok/s]

==================== penalty=1.1 ====================
설정: {'do_sample': True, 'temperature': 0.8, 'top_p': 0.9, 'repetition_penalty': 1.1, 'max_new_tokens': 180}
1. 로컬 LLM 학습 모임에서 함께 즐기고 커뮤니티를 만든다.  
2. 다양한 언어와 놀이를 통해 서로 다른 사람들을 연결합니다.  
3. 기술과 열정을 결합하여 창의성을 발휘할 수 있는 온라인 공간입니다.  
4. 소규모 팀으로 진행하며 성공적인 프로젝트를 만들어 나갈 수 있습니다.  
5. 글쓰기를 위한 교육과 지식을 공유하면서 더 나은 인문적 가치를 창출할 수 있을 것입니다.  
6. 개인과 공동체에 대한 관심을 이끌어내고, 협력과 성장의 기회를 제공할 수 있을 것입니다.
[166 tokens / 5.26 sec / 31.6 tok/s]

==================== penalty=1.3 ====================
설정: {'do_sample': True, 'temperature': 

### 기록 D

**Q1. 반복되는 단어나 문장 구조가 가장 많았던 설정**
> **`repetition_penalty=1.0`**. `로컬 LLM 학습`, `~을 위한`, `모임`과 비슷한 문장 구조가 여섯 항목에서 계속 반복됐다.

**Q2. 다양성과 자연스러움의 균형이 가장 좋았던 설정**
> **`repetition_penalty=1.1`**  
> 동일 표현이 줄고 `커뮤니티`, `프로젝트`, `교육`, `협력과 성장` 등 표현 범위가 넓어졌으며 문장도 대체로 자연스러웠다. 다만 일부 문구는 홍보 대상에서 벗어났다.

**Q3. penalty가 너무 높을 때 발견한 부작용**
> **`1.3`에서는 반복은 줄었지만 의미 연결이 약해지고 `최상의 가격이 가장 높은`, `싱결 커뮤니티` 같은 부자연스러운 표현이 나타났다.** 마지막 문장은 최대 토큰 수에 도달해 잘리기도 했다.

## 7. 능력과 안정성 확인: 시드를 바꿔 세 번 생성하기

샘플링 결과 하나만 보고 모델이 `할 수 있다` 또는 `할 수 없다`고 단정하면 안 됩니다. 같은 설정을 서로 다른 seed로 반복해 **가능한 출력의 범위**와 **성공의 안정성**을 구분합니다.

In [20]:
my_options = {
    "do_sample": True,
    "temperature": 0.8,
    "top_p": 0.9,
    "top_k": 20
}

seed_results = []
for seed in [7, 42, 2026]:
    result = ask(creative_prompt, seed=seed, **my_options)
    seed_results.append(result)
    print(f"\n[seed={seed}]\n{result['answer']}")


[seed=7]
1. 점심  
2. 절심

[seed=42]
1. 점심을 주제로 한 이행시  
2. 점심을 주제로 한 이행시  
3. 점심을 주제로 한 이행시

[seed=2026]
점심을 주제로 '점'과 '심'으로 시작하는 이행시:  
점심은 소리가 빛나는 곳, 심장이 힘들어하는 곳.


### 기록 E

**Q1. seed 2026의 결과는 이행시에 성공했는가?**
> **핵심 형식에는 부분적으로 성공했다.**  
> `점심은...`과 `심장이...`로 두 구절을 연결했으므로 `점 → 심`으로 이어지는 이행시 패턴을 생성했다. 다만 요청 문장을 먼저 반복했고, 두 구절을 정확히 두 줄로 나누지 않았으며, 점심이라는 주제에서도 벗어났다. 따라서 **이행시 패턴 성공**과 **전체 지시 성공**은 구분해야 한다.

**Q2. 그렇다면 이 모델은 이행시를 할 수 있는가?**
> **할 수 있다.** 적어도 seed 2026의 결과는 모델이 이행시의 핵심 패턴을 학습했고 그 패턴을 출력으로 꺼낼 수 있다는 증거다. 토큰화 때문에 원천적으로 불가능한 작업도 아니다.

**Q3. 그런데 다른 seed에서는 왜 실패했는가?**
> 샘플링은 매 단계에서 확률 분포를 따라 토큰을 선택한다. seed가 달라지면 초반 선택이 달라지고, 이후 문맥도 달라져 서로 다른 생성 경로로 들어간다. 이 모델은 올바른 이행시 경로도 가지고 있지만, 요청 반복이나 잘못된 형식에도 높은 확률을 주기 때문에 결과가 불안정하다.

### 핵심 구분

- **능력 보유**: 한 번이라도 이행시 핵심 패턴을 생성할 수 있는가? → **예**
- **전체 지시 준수**: 형식·주제·출력 조건을 모두 지키는가? → **seed 2026도 실패**
- **안정성**: seed가 바뀌어도 높은 비율로 성공하는가? → **현재 세 번의 결과에서는 낮음**

> 한 번의 성공은 **능력이 존재한다는 증거**가 될 수 있지만, 한 번의 성공만으로 **신뢰할 수 있는 성능**까지 입증되지는 않는다.

## 8. 부분 성공과 실패 원인 분석

앞의 실험에서는 요청 반복, 형식 실패, 부분 성공이 모두 나타났습니다. 이를 `모델이 못한다`로 한꺼번에 묶지 않고 다음 네 가지를 나눠 확인합니다.

1. **토큰화**: `점`과 `심`이 한 글자 단위로 표현되는가?
2. **프롬프트 형식**: 모델이 출력 규칙을 더 명확하게 이해하면 결과가 나아지는가?
3. **디코딩 반복**: 반복 패널티를 적용하면 같은 구절에서 빠져나오는가?
4. **성공률**: 올바른 생성 경로가 존재하더라도 얼마나 자주 그 경로를 선택하는가?

In [21]:
# 8-1. '점'과 '심'의 토큰화 확인
for text in ["점", "심", "점심"]:
    token_ids = tok.encode(text, add_special_tokens=False)
    restored = tok.decode(token_ids)
    print(f"{text!r:6} -> {len(token_ids)}개 토큰 / ids={token_ids} / 복원={restored!r}")

'점'    -> 1개 토큰 / ids=[126333] / 복원='점'
'심'    -> 1개 토큰 / ids=[125512] / 복원='심'
'점심'   -> 2개 토큰 / ids=[126333, 125512] / 복원='점심'


`점`과 `심`이 각각 하나의 토큰이고 다시 원래 글자로 복원되므로, 글자가 토큰 중간에서 쪼개져서 생긴 실패는 아닙니다. seed 2026의 부분 성공까지 함께 보면 모델은 이행시 패턴을 생성할 수 있습니다. 이제 질문은 `가능한가?`보다 `얼마나 정확하고 안정적으로 수행하는가?`가 됩니다.

In [22]:
# 8-2. 같은 seed로 기존 프롬프트, 구조화 프롬프트, 반복 억제를 차례로 비교
structured_prompt = """
한국어 2행시를 작성하세요.

규칙:
- 정확히 두 줄만 출력
- 첫째 줄은 반드시 \"점:\"으로 시작
- 둘째 줄은 반드시 \"심:\"으로 시작
- 제목이나 안내문은 출력하지 않음
- 각 줄은 20자 이내

점:
심:
""".strip()

base_options = {
    "do_sample": True,
    "temperature": 0.7,
    "top_p": 0.9,
    "top_k": 20,
    "max_new_tokens": 80,
}

comparison_specs = [
    ("A. 기존 프롬프트", creative_prompt, base_options),
    ("B. 구조화 프롬프트", structured_prompt, base_options),
    ("C. 구조화 + 반복 억제", structured_prompt, {**base_options, "repetition_penalty": 1.1}),
]

failure_analysis_results = []
for label, prompt, options in comparison_specs:
    result = ask(prompt, seed=42, **options)
    failure_analysis_results.append({"label": label, **result})
    print(f"\n{'=' * 16} {label} {'=' * 16}")
    print(result["answer"])


================ A. 기존 프롬프트 ================
1. 점심을 주제로 한 이행시  
2. 점심을 주제로 한 이행시  
3. 점심을 주제로 한 이행시

================ B. 구조화 프롬프트 ================
점:심:

================ C. 구조화 + 반복 억제 ================
점: 심:


### 결과 해석 및 기록

- `점`의 토큰 수: **1개** (`[126333]`).
- `심`의 토큰 수: **1개** (`[125512]`).
- **해석** : 두 글자가 토큰 중간에서 쪼개져 발생한 문제는 아니다.

**Q1. B는 A보다 두 줄 형식을 잘 지켰는가?**
> **아니다.**  
> A의 세 줄 반복 대신 `점:심:`이라는 형식 단서는 출력했지만 줄바꿈과 각 행의 내용이 없어, 이행시 형식을 완성하지 못했다.

**Q2. C는 B보다 동일 구절의 반복이 줄었는가?**
> **판단할 수 없다.**  
> B부터 출력이 매우 짧고 반복 구절이 없어서, C의 repetition penalty 효과를 비교할 충분한 반복 사례가 없다.  
> C는 공백 하나가 추가된 `점: 심:`만 출력했다.

**Q3. B와 C의 결과만 보고 모델이 이행시를 못한다고 결론 내릴 수 있는가?**
> **결론 내릴 수 없다.** seed 2026에서 이미 `점 → 심` 패턴을 생성했으므로 이행시 능력 자체는 확인됐다. B와 C는 모델 역량뿐 아니라, 입력 끝에 `점:`과 `심:`을 모두 미리 둔 프롬프트 설계의 영향도 받았다.

**Q4. 현재 결과에서 확인된 실제 한계는 무엇인가?**
> **전체 조건을 동시에 지키는 성공률과 안정성이 낮다는 점이다.** 모델은 이행시 핵심 패턴을 출력할 수 있지만, 형식·주제·불필요한 설명 금지 조건을 seed가 바뀌어도 일관되게 만족시키지는 못했다.

### 판단 기준

- 글자가 정상적으로 토큰화되면 **토큰 분할이 직접적인 원인일 가능성은 낮습니다.**
- 구조화 프롬프트로 개선되면 **기존 지시가 작은 모델에 충분히 명확하지 않았던 것**입니다.
- 반복 패널티로 반복만 줄고 내용이 나아지지 않으면 **디코딩 문제는 완화됐지만 모델 역량은 그대로인 것**입니다.
- 일부 seed에서 성공하고 다수 seed에서 실패하면 **능력은 존재하지만 그 능력을 안정적으로 끌어내는 확률이 낮다**고 해석합니다.
- 형식·주제·출력 조건을 각각 평가해야 부분 성공을 전체 실패로 뭉뚱그리지 않을 수 있습니다.

> 생성 옵션은 모델이 가진 여러 생성 경로의 선택 확률을 바꿉니다. 한 seed의 성공은 가능한 경로의 존재를 보여주고, 여러 seed의 성공률은 그 경로가 얼마나 안정적으로 선택되는지를 보여줍니다.

## 9. 작업별 추천 설정 만들기

아래 표를 자신의 실험 결과로 완성하세요. 숫자만 적지 말고 결과에서 찾은 근거를 한 문장씩 씁니다.

| 작업 | do_sample | temperature | top_p | top_k | 선택 이유 |
|---|---:|---:|---:|---:|---|
| 감정 분류 | `False` | — | — | — | 정답 후보가 정해진 작업이므로 가장 확률 높은 결과를 고정적으로 생성하는 편이 적합하다. |
| 한 문장 요약 | `False` | — | — | — | 창의성보다 원문 보존과 실행 간 일관성이 중요하므로 greedy를 출발점으로 삼는다. |
| 번역 | `False` | — | — | — | 의미를 임의로 변형할 필요가 없고 같은 입력에 안정적인 번역을 얻는 것이 중요하다. |
| 창작 글쓰기 | `True` | `0.7` | `0.9` | `20` | 이번 실험에서 낮은 값의 반복과 높은 temperature의 무의미한 표현 사이에서 비교적 균형이 좋았던 출발 설정이다. |

> `do_sample=False`일 때 temperature, top_p, top_k는 샘플링에 사용되지 않으므로 `—`로 표시했다. 위 값은 절대적인 정답이 아니라 이번 결과를 바탕으로 정한 **초기 설정**이며, 실제 작업별 평가 데이터로 다시 검증해야 한다.
